# Validation of logical resource estimates

$\renewcommand{\ket}[1]{\left|#1\right\rangle}\renewcommand{\bra}[1]{\left\langle #1\right|}\renewcommand{\braket}[2]{\left\langle #1 \middle| #2 \right\rangle}\renewcommand{\ketbra}[2]{\left|#1\right\rangle\!\left\langle #2\right|}$

This notebook validates the logical-qubit counts and non-Clifford-depth bounds for the current circuit API; the fully phase section is retained as a reference. Each implemented circuit is recursively decomposed into the Clifford+$T$+$R_z$ gate set and compared with the corresponding analytical resource formula.

Logical-qubit counts are checked exactly. The depth formulas are upper bounds, so an implementation is accepted when its measured non-Clifford depth does not exceed the stated bound. Special parameter choices may produce a smaller depth.

**Table of contents**

1. [Import and validation helper](#import-and-validation-helper)
2. [Primitive gates](#primitive-gates)
3. [Multi-controlled NOT](#multi-controlled-not)
4. [Proposal preparation](#proposal-preparation)
5. [Fully phase-based arithmetic](#fully-phase-based-arithmetic)
6. [Fixed-point arithmetic and rotation coin](#fixed-point-arithmetic)
7. [Reflection and accept-path unitary](#reflection-and-accept-path)

<a id="import-and-validation-helper"></a>
## Import and validation helper

The helper below checks the logical-qubit count and computes the non-Clifford depth using the explicit circuit decompositions in `monaqa2`.

In [1]:
import sys

if ".." not in sys.path:
    sys.path.append("..")  # Import the local package without an editable installation.

import numpy as np
from qiskit.circuit import QuantumCircuit
from monaqa2.qiskit.utils_qiskit import get_nc_depth


def ceil_log2(x: int) -> int:
    """Return ceil(log2(x)) for a positive integer."""
    if x < 1:
        raise ValueError("x must be positive.")
    return int(np.ceil(np.log2(x)))


def validate_resource(
    operation,
    expected_num_qubits: int,
    expected_depth: int,
    *,
    verbose: bool = False,
) -> None:
    """Validate an exact qubit count and an upper bound on non-Clifford depth."""
    name = getattr(operation, "name", operation.__class__.__name__)
    actual_num_qubits = operation.num_qubits

    if actual_num_qubits != expected_num_qubits:
        raise ValueError(
            f"{name} uses {actual_num_qubits} qubits; "
            f"expected {expected_num_qubits}."
        )

    if isinstance(operation, QuantumCircuit):
        circuit = operation
    else:
        circuit = QuantumCircuit(expected_num_qubits)
        circuit.append(operation, range(expected_num_qubits))

    actual_depth = get_nc_depth(circuit)
    if actual_depth > expected_depth:
        raise ValueError(
            f"{name} has non-Clifford depth {actual_depth}; "
            f"the analytical upper bound is {expected_depth}."
        )

    if verbose:
        print(
            f"{name}: qubits={actual_num_qubits}, "
            f"depth={actual_depth} <= {expected_depth}"
        )


<a id="primitive-gates"></a>
## Primitive gates

The following checks reproduce Table III of the paper.

| Gate | Logical qubits | Non-Clifford depth |
|---|---:|---:|
| $\mathrm{CCX}$ | $3$ | $3$ |
| $\mathrm{CR}_y(\theta)$ | $2$ | $2$ |
| $\mathrm{CCR}_y(\theta)$ | $3$ | $8$ |
| $G(\theta)$ | $2$ | $2$ |
| $\mathrm{C}G(\theta)$ | $3$ | $14$ |


### Direct validation

The primitive gates are instantiated with generic nonzero angles so that their explicit decompositions contain the expected non-Clifford layers.


In [2]:
from monaqa2.qiskit.primitives import (
    Ccx,
    Cry,
    Ccry,
    GivensRotation,
    ControlledGivensRotation,
)

validate_resource(Ccx(), 3, 3, verbose=True)
validate_resource(Cry(0.1), 2, 2, verbose=True)
validate_resource(Ccry(0.1), 3, 8, verbose=True)
validate_resource(GivensRotation(0.3, 0.7), 2, 2, verbose=True)
validate_resource(
    ControlledGivensRotation(0.3, 0.7),
    3,
    14,
    verbose=True,
)


ccx: qubits=3, depth=3 <= 3
cry: qubits=2, depth=2 <= 2
ccry: qubits=3, depth=8 <= 8
givens: qubits=2, depth=2 <= 2
c_givens: qubits=3, depth=14 <= 14


<a id="multi-controlled-not"></a>
## Multi-controlled NOT

The two-clean-ancilla construction `synth_mcx_2_clean_kg24` implements an $m$-controlled NOT using $m+3$ logical qubits. For $m\geq3$, its non-Clifford depth is upper-bounded by

$$
14\left\lceil\log_2m\right\rceil-10.
$$

The following cell checks the bound over consecutive and power-of-two control sizes.


In [3]:
from qiskit.synthesis.multi_controlled.mcx_synthesis import (
    synth_mcx_2_clean_kg24,
)

control_sizes = list(range(3, 16)) + [2**power for power in range(4, 16)]

for num_controls in control_sizes:
    expected_depth = 14 * ceil_log2(num_controls) - 10
    operation = synth_mcx_2_clean_kg24(num_controls)

    print(f"m={num_controls:5d} | ", end="")
    validate_resource(
        operation,
        expected_num_qubits=num_controls + 3,
        expected_depth=expected_depth,
        verbose=True,
    )


m=    3 | mcx_logn_depth: qubits=6, depth=11 <= 18
m=    4 | mcx_logn_depth: qubits=7, depth=18 <= 18
m=    5 | mcx_logn_depth: qubits=8, depth=22 <= 32
m=    6 | mcx_logn_depth: qubits=9, depth=27 <= 32
m=    7 | mcx_logn_depth: qubits=10, depth=27 <= 32
m=    8 | mcx_logn_depth: qubits=11, depth=28 <= 32
m=    9 | mcx_logn_depth: qubits=12, depth=30 <= 46
m=   10 | mcx_logn_depth: qubits=13, depth=32 <= 46
m=   11 | mcx_logn_depth: qubits=14, depth=36 <= 46
m=   12 | mcx_logn_depth: qubits=15, depth=36 <= 46
m=   13 | mcx_logn_depth: qubits=16, depth=40 <= 46
m=   14 | mcx_logn_depth: qubits=17, depth=42 <= 46
m=   15 | mcx_logn_depth: qubits=18, depth=44 <= 46
m=   16 | mcx_logn_depth: qubits=19, depth=46 <= 46
m=   32 | mcx_logn_depth: qubits=35, depth=56 <= 60
m=   64 | mcx_logn_depth: qubits=67, depth=74 <= 74
m=  128 | mcx_logn_depth: qubits=131, depth=84 <= 88
m=  256 | mcx_logn_depth: qubits=259, depth=102 <= 102
m=  512 | mcx_logn_depth: qubits=515, depth=110 <= 116
m= 1024 |

<a id="proposal-preparation"></a>
## Proposal preparation

The first three rows reproduce Table IV of the paper. The randomized row checks the additional coherent midpoint-grid implementation with $2^k$ values of $t$ and $2^k$ values of $\gamma$.

| Proposal block | Logical qubits | Non-Clifford depth bound |
|---|---:|---:|
| Uniform proposal | $2n$ | $0$ |
| Local proposal | $2n$ | $2\lceil\log_2n\rceil$ |
| Hamiltonian-simulation proposal | $2n$ | $r(n+3)$ |
| Randomized Hamiltonian-simulation proposal | $2n(k+1)$ | $r(n+1)(1+4k+4k^2)+(r+1)(1+4k+6k^2)$ |

`ProposalQemc` is the fixed-parameter Hamiltonian-simulation proposal. Its $r(n+3)$ depth is the paper's bound; an even $n$ can need one fewer $ZZ$ matching layer.

`ProposalQemcRandomized` adds $2k$ parameter qubits and $2k(n-1)$ fanout work qubits. For its compiled gates, a single Ising layer has depth at most $1+4k+4k^2$ (ordinary, singly controlled, and doubly controlled $R_z$ rotations). A mixer layer has depth at most $1+4k+6k^2$ for the corresponding $R_x$ rotations. There are at most $n$ $ZZ$ matchings plus one local-field layer per Trotter step. Combining adjacent mixer half steps leaves $r+1$ mixer layers. The controlled-rotation costs $1,2,4$ for $R_z$ and $1,2,6$ for $R_x$ are measured after decomposition by the notebook's `get_nc_depth` helper. Fanout uses Clifford CNOTs and adds zero non-Clifford depth.

Randomized checks use small $n$, $k$, and $r$ to keep circuit decomposition practical; the bound holds for general positive $r$.


In [4]:
from monaqa2.qiskit.proposal_uniform import ProposalUniform
from monaqa2.qiskit.proposal_local import ProposalLocal
from monaqa2.qiskit.proposal_qemc import ProposalQemc
from monaqa2.qiskit.proposal_qemc_randomized import ProposalQemcRandomized


def validate_proposal_resources(n: int, num_trotter_steps: int = 50) -> None:
    """Validate the proposal resource bounds, including small randomized grids."""
    if n < 2:
        raise ValueError("Use n >= 2.")

    h = np.ones(n)
    J = np.ones((n, n)) - np.eye(n)
    gamma = np.ones(n)
    evolution_time = 1.0

    validate_resource(
        ProposalUniform(n),
        expected_num_qubits=2 * n,
        expected_depth=0,
    )
    validate_resource(
        ProposalLocal(n, k=1),
        expected_num_qubits=2 * n,
        expected_depth=2 * ceil_log2(n),
    )
    validate_resource(
        ProposalQemc(
            n,
            h,
            J,
            gamma,
            evolution_time,
            num_trotter_steps=num_trotter_steps,
        ),
        expected_num_qubits=2 * n,
        expected_depth=num_trotter_steps * (n + 3),
        verbose=True
    )

    if n <= 6:
        for k in (1, 2, 3):
            r = num_trotter_steps
            z_depth = 1 + 4 * k + 4 * k**2
            x_depth = 1 + 4 * k + 6 * k**2
            validate_resource(
                ProposalQemcRandomized(n, h, J, k, num_trotter_steps=r),
                expected_num_qubits=2 * n * (k + 1),
                expected_depth=r * (n + 1) * z_depth + (r + 1) * x_depth,
                verbose=True,
            )

    print(f"All proposal checks passed for n={n}.")


proposal_sizes = list(range(2, 16)) + [2**power for power in range(4, 8)]
for n in proposal_sizes:
    # r=2 checks the merged mixer layers without a costly randomized decomposition.
    validate_proposal_resources(n, num_trotter_steps=2 if n <= 4 else 50)

ProposalQemc(n=2,trotter_steps=2): qubits=4, depth=8 <= 10
ProposalQemcRandomized: qubits=8, depth=69 <= 87
ProposalQemcRandomized: qubits=12, depth=199 <= 249
ProposalQemcRandomized: qubits=16, depth=397 <= 495
All proposal checks passed for n=2.
ProposalQemc(n=3,trotter_steps=2): qubits=6, depth=12 <= 12
ProposalQemcRandomized: qubits=12, depth=105 <= 105
ProposalQemcRandomized: qubits=18, depth=299 <= 299
ProposalQemcRandomized: qubits=24, depth=593 <= 593
All proposal checks passed for n=3.
ProposalQemc(n=4,trotter_steps=2): qubits=8, depth=12 <= 14
ProposalQemcRandomized: qubits=16, depth=105 <= 123
ProposalQemcRandomized: qubits=24, depth=299 <= 349
ProposalQemcRandomized: qubits=32, depth=593 <= 691
All proposal checks passed for n=4.
ProposalQemc(n=5,trotter_steps=50): qubits=10, depth=400 <= 400
ProposalQemcRandomized: qubits=20, depth=3261 <= 3261
ProposalQemcRandomized: qubits=30, depth=9183 <= 9183
ProposalQemcRandomized: qubits=40, depth=18117 <= 18117
All proposal checks 


KeyboardInterrupt


KeyboardInterrupt



<a id="fully-phase-based-arithmetic"></a>
## Fully phase-based arithmetic

The following table reproduces Table V of the paper. We use $\ell_x=\lceil\log_2x\rceil$.

| Component | Logical qubits | Non-Clifford depth |
|---|---:|---:|
| $\mathrm{PREPARE}$ | $6n$ | $60n-56$ |
| $\mathrm{SELECT}$ | $8n$ | $0$ |
| controlled-$\mathrm{SELECT}$ | $10n$ | $6n+9$ |
| $R_0$ | $6n+2$ | $14\ell_{6n}-10$ |
| controlled-$R_0$ | $6n+3$ | $14\ell_{6n}-10$ |
| $Q_{\mathrm{ph}}$ | $8n+2$ | $120n+14\ell_{6n}-122$ |
| controlled-$Q_{\mathrm{ph}}$ | $10n+2$ | $126n+14\ell_{6n}-113$ |
| Fully phase arithmetic | $10n+2$ | $3d_{\mathrm{ph}}(126n+14\ell_{6n}-113)+3(2d_{\mathrm{ph}}+1)$ |

The validation uses a fixed test degree $d_{\mathrm{ph}}$. The resource formulas are independent of the numerical values of the SK coefficients, provided the circuit structure is unchanged.


In [6]:
from monaqa2.qiskit.arithmetic_fully_phase import (
    PrepareDeltaHamiltonian,
    SelectDeltaHamiltonian,
    ControlledSelectDeltaHamiltonian,
    ReflectionZero,
    ControlledReflectionZero,
    QubitizedDeltaHamiltonian,
    ControlledQubitizedDeltaHamiltonian,
    FullyPhaseArithmetic,
)


def validate_fully_phase_resources(
    n: int,
    *,
    d_ph: int = 4,
    verbose: bool = False,
) -> None:
    """Validate all rows of Table V."""
    if n < 2:
        raise ValueError("Use n >= 2.")

    h = np.ones(n)
    J = np.ones((n, n)) - np.eye(n)
    beta = 1.0
    eps_ops = 1e-3
    ell_6n = ceil_log2(6 * n)

    validate_resource(PrepareDeltaHamiltonian(n, h, J), 6 * n, 60 * n - 56, verbose=verbose)
    validate_resource(SelectDeltaHamiltonian(n, h, J), 8 * n, 0, verbose=verbose)
    validate_resource(ControlledSelectDeltaHamiltonian(n, h, J), 10 * n, 6 * n + 9, verbose=verbose)
    validate_resource(ReflectionZero(6 * n), 6 * n + 2, 14 * ell_6n - 10, verbose=verbose)
    validate_resource(ControlledReflectionZero(6 * n), 6 * n + 3, 14 * ell_6n - 10, verbose=verbose)
    validate_resource(QubitizedDeltaHamiltonian(n, h, J), 8 * n + 2, 120 * n + 14 * ell_6n - 122, verbose=verbose)
    validate_resource(ControlledQubitizedDeltaHamiltonian(n, h, J), 10 * n + 2, 126 * n + 14 * ell_6n - 113, verbose=verbose)
    validate_resource(
        FullyPhaseArithmetic(
            n,
            h,
            J,
            beta=beta,
            eps_ops=eps_ops,
            degree=d_ph,
            is_mocked_construction=False,
            is_mocked_angles=True,
        ),
        expected_num_qubits=10 * n + 2,
        expected_depth=3 * d_ph * (126 * n + 14 * ell_6n - 113) + 3 * (2 * d_ph + 1),
        verbose=verbose,
    )

    print(f"All fully phase-based checks passed for n={n}.")


fully_phase_sizes = list(range(2, 16)) + [2**power for power in range(4, 8)]
for n in fully_phase_sizes:
    validate_fully_phase_resources(n)


All fully phase-based checks passed for n=2.
All fully phase-based checks passed for n=3.
All fully phase-based checks passed for n=4.
All fully phase-based checks passed for n=5.
All fully phase-based checks passed for n=6.
All fully phase-based checks passed for n=7.
All fully phase-based checks passed for n=8.
All fully phase-based checks passed for n=9.
All fully phase-based checks passed for n=10.
All fully phase-based checks passed for n=11.
All fully phase-based checks passed for n=12.
All fully phase-based checks passed for n=13.
All fully phase-based checks passed for n=14.
All fully phase-based checks passed for n=15.
All fully phase-based checks passed for n=16.
All fully phase-based checks passed for n=32.


KeyboardInterrupt: 

<a id="fixed-point-arithmetic"></a>
## Fixed-point arithmetic and rotation coin

The shared arithmetic primitives still come from `arithmetic_hybrid`; the complete new gate and its rotation coin come from `arithmetic_fixed_point`. Use $W=F+1$, $M=n(n+1)/2$, and $s_{\mathrm{SK}}=\lceil\log_{3/2}(2M)\rceil$.

| Component | Logical qubits | Non-Clifford depth bound |
|---|---:|---:|
| Conditional terms loader | $MW+2M-n$ | $0$ |
| Three-to-two compressor | $5$ | $9$ |
| Wallace-tree adder | $6MW-2W-2M+1$ | $18s_{\mathrm{SK}}+18W$ |
| Energy-difference block | $2n+6MW-2W-2M+1$ | $18s_{\mathrm{SK}}+18W$ |
| Positive-part selector | $3W$ | $3$ |
| Independent rotation coin | $2W$ | $D_{\mathrm{rot}}$ |
| Complete fixed-point arithmetic | $2n+6MW-2M+W+1$ | $2(18s_{\mathrm{SK}}+18W)+6+D_{\mathrm{rot}}$ |

For nonzero normalization, $D_{\mathrm{rot}}=2$ at positive beta and zero at beta=0. There are at most 2W arbitrary Rz rotations, scheduled in two parallel layers. No polynomial-degree or cutoff parameters enter these bounds. The checks include both beta=0 and positive beta, plus a comparison with the public runtime estimator's selected precision.

In [5]:
from monaqa2.qiskit.arithmetic_hybrid import (
    ThreeTwoCompressor,
    WallaceTreeAdder,
    ConditionalTermsLoader,
    DeltaEnergy,
    PositivePartSelector,
)
from monaqa2.qiskit.arithmetic_fixed_point import FixedPointArithmetic, RotationBoltzmannCoin
from monaqa2.data.runtime import fixed_point_arithmetic_circuit


def validate_fixed_point_arithmetic_resources(n: int, W: int, beta: float = 1.0) -> None:
    """Check shared arithmetic and the complete rotation-based coin."""
    if n < 2 or W < 2:
        raise ValueError("Use n >= 2 and W >= 2.")
    M = n * (n + 1) // 2
    F = W - 1
    s_sk = int(np.ceil(np.log(2 * M) / np.log(1.5)))
    h = np.zeros(n)
    h[:2] = [0.5, -0.25]
    J = np.zeros((n, n))
    J[0, 1] = J[1, 0] = 0.25
    alpha = ConditionalTermsLoader._alpha(h, J)
    delta_depth = 18 * s_sk + 18 * W
    rotation_depth = 0 if beta == 0 else 2

    validate_resource(ConditionalTermsLoader(n, h, J, F, invert_coefficients=False), M * W + 2 * M - n, 0)
    validate_resource(ThreeTwoCompressor(), 5, 9)
    validate_resource(WallaceTreeAdder(2 * M, W), 6 * M * W - 2 * W - 2 * M + 1, delta_depth)
    validate_resource(DeltaEnergy(n, h, J, F), 2 * n + 6 * M * W - 2 * W - 2 * M + 1, delta_depth)
    validate_resource(PositivePartSelector(W), 3 * W, 3)
    validate_resource(RotationBoltzmannCoin(W, beta, alpha), 2 * W, rotation_depth)
    validate_resource(
        FixedPointArithmetic(n, h, J, F, beta),
        2 * n + 6 * M * W - 2 * M + W + 1,
        2 * delta_depth + 6 + rotation_depth,
        verbose=True,
    )


for n, W in [(2, 4), (3, 6), (4, 8), (6, 10), (8, 12), (12, 14)]:
    for beta in (0.0, 1.0):
        validate_fixed_point_arithmetic_resources(n, W, beta)

# Match the circuit's explicit F to the public estimator's retained precision rule.
n, eps_W, beta = 3, 0.1, 1.0
F = int(np.ceil(3.5 * (1 + np.log2(n) + np.log2(1 / eps_W))))
expected_depth, expected_qubits = fixed_point_arithmetic_circuit(n, eps_W, beta)
validate_resource(
    FixedPointArithmetic(n, np.array([0.5, -0.25, 0.125]), np.zeros((n, n)), F, beta),
    int(expected_qubits),
    int(expected_depth),
    verbose=True,
)

FixedPointArithmetic(2,3): qubits=75, depth=172 <= 330
FixedPointArithmetic(2,3): qubits=75, depth=172 <= 332
FixedPointArithmetic(3,5): qubits=217, depth=302 <= 474
FixedPointArithmetic(3,5): qubits=217, depth=302 <= 476
FixedPointArithmetic(4,7): qubits=477, depth=432 <= 582
FixedPointArithmetic(4,7): qubits=477, depth=432 <= 584
FixedPointArithmetic(6,9): qubits=1241, depth=530 <= 726
FixedPointArithmetic(6,9): qubits=1241, depth=530 <= 728
FixedPointArithmetic(8,11): qubits=2549, depth=660 <= 834
FixedPointArithmetic(8,11): qubits=2549, depth=660 <= 836
FixedPointArithmetic(12,13): qubits=6435, depth=792 <= 978
FixedPointArithmetic(12,13): qubits=6435, depth=792 <= 980
FixedPointArithmetic(3,21): qubits=809, depth=782 <= 1052


<a id="reflection-and-accept-path"></a>
## Reflection and accept-path unitary

The fixed-point coin leaves W rotation ancillas as its persistent coin register. Its complete arithmetic workspace is uncomputed before these gates are applied. Pass `coins=W`, and use the all-zero coin state as the accepting subspace.

| Component | Logical qubits | Non-Clifford depth bound |
|---|---:|---:|
| Reflection | $2n+W+2$ | $14\lceil\log_2(n+W-1)\rceil-10$ |
| Accept-path unitary | $3n+W$ | $28\lceil\log_2W\rceil-11$ |

These rows apply for $n\geq3$, $W\geq2$. The accept-path unitary computes and uncomputes its zero-coin flag, with a parallel controlled-swap stage of non-Clifford depth nine.

In [ ]:
from monaqa2.qiskit.reflection_updated import Reflection
from monaqa2.qiskit.accept_path_updated import AcceptPath


def validate_reflection_accept_path_resources(n: int, w: int) -> None:
    """Validate reflection and accept path for W=w rotation coins."""
    if n < 3:
        raise ValueError("Use n >= 3.")
    if w < 2:
        raise ValueError("Use w >= 2.")

    coin_qubits = w
    validate_resource(
        Reflection(n, coins=coin_qubits),
        2 * n + w + 2,
        14 * ceil_log2(n + w - 1) - 10,
    )
    validate_resource(
        AcceptPath(n, coins=coin_qubits),
        3 * n + w,
        28 * ceil_log2(w) - 11,
    )
    print(f"Fixed-point reflection/accept-path checks passed for n={n}, w={w}.")


reflection_test_cases = [
    (3, 2),
    (3, 6),
    (4, 8),
    (8, 8),
    (8, 15),
    (16, 15),
    (16, 31),
    (32, 31),
    (32, 63),
    (64, 63),
]

for test_case in reflection_test_cases:
    validate_reflection_accept_path_resources(*test_case)